In [1]:
import pandas as pd

# Load the CSV file into a DataFrame
ball_by_ball_data = pd.read_csv('./ipl-dataset-2008-to-2025/ball_by_ball_data.csv')
ipl_matches_data = pd.read_csv('./ipl-dataset-2008-to-2025/ipl_matches_data.csv')
players_data = pd.read_csv('./ipl-dataset-2008-to-2025/players-data-updated.csv')
teams_data = pd.read_csv('./ipl-dataset-2008-to-2025/teams_data.csv')
# View the first 5 rows
# ball_by_ball_data.head()

In [2]:
ipl_matches_data.head()

,match_id,season_id,balls_per_over,city,match_date,event_name,match_number,gender,match_type,format,...,venue,toss_winner,team1,team2,toss_decision,match_winner,win_by_runs,win_by_wickets,player_of_match,result
0,335982,2008,6,Bengaluru,2008-04-18,Indian Premier League,1.0,male,T20,T20,...,M Chinnaswamy Stadium,1,1,6,field,6,140.0,NaN,46.0,win
1,1082591,2017,6,Hyderabad,2017-04-05,Indian Premier League,1.0,male,T20,T20,...,"Rajiv Gandhi International Stadium, Uppal",1,2,1,field,2,35.0,NaN,15.0,win
2,1082592,2017,6,Pune,2017-04-06,Indian Premier League,2.0,male,T20,T20,...,Maharashtra Cricket Association Stadium,4,4,3,field,4,NaN,7.0,36.0,win
3,1082593,2017,6,Rajkot,2017-04-07,Indian Premier League,3.0,male,T20,T20,...,Saurashtra Cricket Association Stadium,6,5,6,field,6,NaN,10.0,57.0,win
4,1082594,2017,6,Indore,2017-04-08,Indian Premier League,4.0,male,T20,T20,...,Holkar Cricket Stadium,494,494,4,field,494,NaN,6.0,71.0,win


In [3]:
teams_data.head(15)

,team_id,team_name
0,1,Royal Challengers Bangalore
1,2,Sunrisers Hyderabad
2,3,Mumbai Indians
3,4,Rising Pune Supergiant
4,5,Gujarat Lions
5,6,Kolkata Knight Riders
6,129,Chennai Super Kings
7,134,Rajasthan Royals
8,252,Delhi Capitals
9,494,Punjab Kings


In [4]:
ball_by_ball_data.head()

,season_id,match_id,batter,bowler,non_striker,team_batting,team_bowling,over_number,ball_number,batter_runs,...,is_bye,is_penalty,wide_ball_runs,no_ball_runs,leg_bye_runs,bye_runs,penalty_runs,wicket_kind,is_super_over,innings
0,2008,335982,SC Ganguly,P Kumar,BB McCullum,6,1,0,0,0,...,False,False,0,0,1,0,0,NaN,False,1
1,2008,335982,BB McCullum,P Kumar,SC Ganguly,6,1,0,1,0,...,False,False,0,0,0,0,0,NaN,False,1
2,2008,335982,BB McCullum,P Kumar,SC Ganguly,6,1,0,2,0,...,False,False,1,0,0,0,0,NaN,False,1
3,2008,335982,BB McCullum,P Kumar,SC Ganguly,6,1,0,3,0,...,False,False,0,0,0,0,0,NaN,False,1
4,2008,335982,BB McCullum,P Kumar,SC Ganguly,6,1,0,4,0,...,False,False,0,0,0,0,0,NaN,False,1


In [5]:
ball_by_ball_data.columns.tolist()

['season_id',
 'match_id',
 'batter',
 'bowler',
 'non_striker',
 'team_batting',
 'team_bowling',
 'over_number',
 'ball_number',
 'batter_runs',
 'extras',
 'total_runs',
 'batsman_type',
 'bowler_type',
 'player_out',
 'fielders_involved',
 'is_wicket',
 'is_wide_ball',
 'is_no_ball',
 'is_leg_bye',
 'is_bye',
 'is_penalty',
 'wide_ball_runs',
 'no_ball_runs',
 'leg_bye_runs',
 'bye_runs',
 'penalty_runs',
 'wicket_kind',
 'is_super_over',
 'innings']

In [6]:
def calculate_batter_performance_metrics(matchup):
    runs = matchup['batter_runs'].sum()
    number_of_fours = (matchup['batter_runs'] == 4).sum()
    number_of_sixes = (matchup['batter_runs'] == 6).sum()
    print(f"number_of_fours : {number_of_fours}")
    print(f"number_of_six : {number_of_sixes}")
    balls = len(matchup)
    outs = matchup['player_out'].count() # or however your 'out' column is named
    batter_average = runs / outs
    strike_rate = (runs / balls) * 100 if balls > 0 else 0
    boundry_percentage = (((number_of_fours * 4) + (number_of_sixes * 6)) / runs) * 100

    # Use outs + 1 to avoid division by zero
    # impact = strike_rate / ((balls / (outs + 1)) + 1) 
    impact = (runs / outs + 1) * (strike_rate / 100)
    
    return {
        "boundary_percentage": round(boundry_percentage, 2).item(),
        "average": round(batter_average, 2).item(),
        "strike_rate": round(strike_rate, 2).item(),
        "impact_score": round(impact, 2).item()
    }

In [7]:
def calculate_bowler_performance_metrics(matchup):
    number_of_balls_bowled = len(matchup)
    number_of_no_balls = matchup['is_no_ball'].sum()
    number_of_wide_balls = matchup['is_wide_ball'].sum()
    batter_runs = matchup['batter_runs'].sum()
    no_ball_runs = matchup['no_ball_runs'].sum()
    wide_ball_runs = matchup['wide_ball_runs'].sum()
    wickets = matchup['player_out'].count() # or however your 'out' column is named
    runs_concedded = batter_runs + no_ball_runs + wide_ball_runs
    
    legal_balls_bowled = number_of_balls_bowled - number_of_no_balls - number_of_wide_balls
    economy_rate = (runs_concedded / legal_balls_bowled) * 6
    bowler_average = runs_concedded / wickets
    strike_rate = legal_balls_bowled / wickets

    # Use outs + 1 to avoid division by zero
    # impact = (wickets+1) * 100 / (runs_concedded + number_of_balls_bowled)
    impact = (wickets) + (number_of_balls_bowled - runs_concedded) / (number_of_balls_bowled/6)
    
    print(f"number_of_balls_bowled : {number_of_balls_bowled} ")
    print(f"wickets : {wickets} ")
    print(f"runs_concedded : {runs_concedded} ")
    return {
        "economy": round(economy_rate, 2).item(),
        "average": round(bowler_average, 2).item(),
        "strike_rate": strike_rate,
        "impact_score": round(impact, 2).item()
    }

In [8]:
def batter_vs_bowler_stats(df, batter, bowler):
    # Use parentheses and & for multiple conditions
    matchup = df[(df['batter'] == batter) & (df['bowler'] == bowler)]
    stats = {
        "type": "batter_vs_bowler_stats",
        "batter": batter,
        "bowler": bowler,
        "stats" : calculate_batter_performance_metrics(matchup)
    }
    return stats


In [9]:
print(batter_vs_bowler_stats(ball_by_ball_data, 'MS Dhoni', 'JJ Bumrah'))
print(batter_vs_bowler_stats(ball_by_ball_data, 'V Kohli', 'JJ Bumrah'))

number_of_fours : 3
number_of_six : 2
{'type': 'batter_vs_bowler_stats', 'batter': 'MS Dhoni', 'bowler': 'JJ Bumrah', 'stats': {'boundary_percentage': 38.71, 'average': 12.4, 'strike_rate': 91.18, 'impact_score': 12.22}}
number_of_fours : 16
number_of_six : 6
{'type': 'batter_vs_bowler_stats', 'batter': 'V Kohli', 'bowler': 'JJ Bumrah', 'stats': {'boundary_percentage': 64.52, 'average': 31.0, 'strike_rate': 149.04, 'impact_score': 47.69}}


In [10]:
def batter_vs_team_stats(df, batter, team):
    # Use parentheses and & for multiple conditions
    matchup = df[(df['batter'] == batter) & (df['team_bowling'] == team)]
    stats = {
        "type": "batter_vs_team_stats",
        "batter": batter,
        "opposition_team": team,
        "stats" : calculate_batter_performance_metrics(matchup)
    }
    return stats

In [11]:
print(batter_vs_team_stats(ball_by_ball_data, 'V Kohli', 3))

number_of_fours : 79
number_of_six : 34
{'type': 'batter_vs_team_stats', 'batter': 'V Kohli', 'opposition_team': 3, 'stats': {'boundary_percentage': 56.09, 'average': 31.97, 'strike_rate': 124.93, 'impact_score': 41.18}}


In [12]:
def batter_at_venue_stats(df, matches_df, batter, city):
    # Use parentheses and & for multiple conditions
    selected_matches =  matches_df[matches_df['city'] == city]['match_id']
    matchup = df[(df['batter'] == batter) & ball_by_ball_data['match_id'].isin(selected_matches)]
    stats = {
        "type": "batter_at_venue_stats",
        "batter": batter,
        "city": city,
        "stats" : calculate_batter_performance_metrics(matchup)
    }
    return stats

In [13]:
def bowler_at_venue_stats(df, matches_df, bowler, city):
    # Use parentheses and & for multiple conditions
    selected_matches =  matches_df[matches_df['city'] == city]['match_id']
    matchup = df[(df['bowler'] == bowler) & ball_by_ball_data['match_id'].isin(selected_matches)]
    return calculate_bowler_performance_metrics(matchup)

In [19]:
print(batter_at_venue_stats(ball_by_ball_data, ipl_matches_data, 'V Kohli', 'Bengaluru'))
# print(batter_at_venue_stats(ball_by_ball_data, ipl_matches_data, 'V Kohli', 'Bangalore'))
# print(batter_at_venue_stats(ball_by_ball_data, ipl_matches_data, 'V Kohli', 'Bangalore'))
# print(batter_at_venue_stats(ball_by_ball_data, ipl_matches_data, 'V Kohli', 'Bangalore'))
# print(batter_at_venue_stats(ball_by_ball_data, ipl_matches_data, 'JJ Bumrah', 'Bangalore'))

number_of_fours : 293
number_of_six : 137
{'type': 'batter_at_venue_stats', 'batter': 'V Kohli', 'city': 'Bengaluru', 'stats': {'boundary_percentage': 62.27, 'average': 38.12, 'strike_rate': 139.83, 'impact_score': 54.7}}


In [ ]:
print(bowler_at_venue_stats(ball_by_ball_data, ipl_matches_data, 'JJ Bumrah', 'Mumbai'))

number_of_balls_bowled : 1283 
wickets : 72 
runs_concedded : 1530 
{'economy': 7.36, 'average': 21.25, 'strike_rate': np.float64(17.319444444444443), 'impact_score': 70.84}


In [ ]:
print(bowler_at_venue_stats(ball_by_ball_data, ipl_matches_data, 'Mohammed Siraj', 'Mumbai'))

number_of_balls_bowled : 369 
wickets : 11 
runs_concedded : 550 
{'economy': 9.48, 'average': 50.0, 'strike_rate': np.float64(31.636363636363637), 'impact_score': 8.06}


In [ ]:
print(bowler_at_venue_stats(ball_by_ball_data, ipl_matches_data, 'AB Dinda', 'Mumbai'))

number_of_balls_bowled : 97 
wickets : 7 
runs_concedded : 161 
{'economy': 10.73, 'average': 23.0, 'strike_rate': np.float64(12.857142857142858), 'impact_score': 3.04}


In [ ]:
team_id = teams_data[teams_data['team_name'] == 'Mumbai Indians']['team_id'].item()

In [ ]:
team_id

3

In [17]:
match_scores = ball_by_ball_data.groupby(['match_id', 'batter'])['batter_runs'].sum().reset_index()

In [18]:
match_scores

,match_id,batter,batter_runs
0,335982,AA Noffke,9
1,335982,B Akhil,0
2,335982,BB McCullum,158
3,335982,CL White,6
4,335982,DJ Hussey,12
...,...,...,...
17633,1485779,SS Iyer,53
17634,1485779,Sameer Rizvi,58
17635,1485779,Sediqullah Atal,22
17636,1485779,Shashank Singh,11
